# Libraries

In [0]:
import os
import requests

from azure.storage.blob import ContainerClient

from src.config.settings import (
    STORAGE_ACCOUNT_NAME,
    CONTAINER_NAME,
    SECRET_SCOPE,
    SECRET_KEY,
    LOCAL_LANDING_PATH,
    AZURE_LANDING_PREFIX,
    SOURCE_BASE_URL,
    SOURCE_FILES
)

# Functions

In [0]:
def get_sas_token() -> str:
    """
    Recupera o SAS Token salvo no Databricks Secret Scope.
    """

    sas_token = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key=SECRET_KEY
    )

    if sas_token.startswith("?"):
        sas_token = sas_token[1:]

    return sas_token


def get_container_client() -> ContainerClient:
    """
    Cria o client do Azure Blob Storage usando SAS Token.
    """

    account_url = f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net"

    return ContainerClient(
        account_url=account_url,
        container_name=CONTAINER_NAME,
        credential=get_sas_token()
    )


def create_local_landing_directory() -> None:
    """
    Cria a pasta local usada como Landing temporária.

    """
    os.makedirs(LOCAL_LANDING_PATH, exist_ok=True)


def blob_exists(container_client: ContainerClient, blob_name: str) -> bool:
    """
    Verifica se o arquivo já existe no Azure Blob Storage.
    """

    blob_client = container_client.get_blob_client(blob_name)

    return blob_client.exists()


def upload_file_to_azure(
    container_client: ContainerClient,
    local_file_path: str,
    blob_name: str
) -> None:
    """
    Faz upload de um arquivo local para o Azure Blob Storage.
    """

    blob_client = container_client.get_blob_client(blob_name)

    with open(local_file_path, "rb") as file:
        blob_client.upload_blob(file, overwrite=True)

    print(f"Arquivo enviado para Azure Landing: {blob_name}")


def download_file_to_local(url: str, local_file_path: str) -> None:
    """
    Faz download do arquivo da NYC TLC para a Landing local.
    Função criada apenas devido ao processamento limitado do Community Edition, 
    """

    print(f"Iniciando download: {url}")

    response = requests.get(url, stream=True, timeout=300)

    if response.status_code != 200:
        raise Exception(
            f"Erro ao baixar arquivo: {url}. "
            f"Status code: {response.status_code}"
        )

    with open(local_file_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    file_size_mb = round(os.path.getsize(local_file_path) / 1024 / 1024, 2)

    if file_size_mb <= 0:
        raise ValueError(f"Arquivo baixado está vazio: {local_file_path}")

    print(f"Arquivo baixado localmente: {local_file_path} | {file_size_mb} MB")


def ingestion_to_landing() -> None:
    """
    Baixa os arquivos Yellow Taxi e Green Taxi de janeiro a maio de 2023,
    salva localmente e envia os arquivos originais para a Landing Zone no Azure Blob.
    """

    create_local_landing_directory()

    container_client = get_container_client()

    for file_name in SOURCE_FILES:
        url = f"{SOURCE_BASE_URL}/{file_name}"

        local_file_path = os.path.join(LOCAL_LANDING_PATH, file_name)
        blob_name = f"{AZURE_LANDING_PREFIX}/{file_name}"

        if os.path.exists(local_file_path) and os.path.getsize(local_file_path) > 0:
            print(f"Arquivo já existe localmente. Pulando download: {local_file_path}")
        else:
            download_file_to_local(url, local_file_path)

        if blob_exists(container_client, blob_name):
            print(f"Arquivo já existe no Azure Landing. Pulando upload: {blob_name}")
        else:
            upload_file_to_azure(container_client, local_file_path, blob_name)

    print("Ingestão para Landing Zone concluída com sucesso.")